# N11 — Overtake Probability: EDA & Labeling

The goal of this notebook is to build the labeled dataset for the overtake probability
model (N12). We work at the level of **car pairs (X, Y)** per lap: X is the chasing
driver, Y is the car directly ahead. For each pair and each lap we determine whether
a real on-track overtake occurred.

The output is a Parquet file with contextual features and a binary `overtake` label,
ready for LightGBM training in N12.

**Data sources:**
- FastF1: positions, lap times, compound, pit stops, track status (2023–2025)
- OpenF1 `/v1/intervals`: real-time gap to the car ahead, aggregated per lap

**Exports:**
- EDA plots → `notebooks/strategy/overtake_probability/outputs/`
- Labeled dataset → `data/processed/overtake_labeled/`


---

## Step 0 — Setup

Standard imports, repo root resolution, and path definitions. FastF1 cache is pointed
at the existing `data/cache/fastf1` directory so we reuse already-downloaded sessions.


In [1]:
# ── Step 0 — Setup ────────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

import pathlib
import requests
import time

import fastf1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# ── Repo root ─────────────────────────────────────────────────────────────────
repo_root = pathlib.Path.cwd()
while not (repo_root / ".git").exists():
    repo_root = repo_root.parent

# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUTS   = repo_root / "notebooks" / "strategy" / "overtake_probability" / "outputs"
PROCESSED = repo_root / "data" / "processed" / "overtake_labeled"
CACHE     = repo_root / "data" / "cache" / "fastf1"

OUTPUTS.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# ── FastF1 cache ──────────────────────────────────────────────────────────────
fastf1.Cache.enable_cache(str(CACHE))

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
sns.set_palette("tab10")

print("repo_root :", repo_root)
print("OUTPUTS   :", OUTPUTS)
print("PROCESSED :", PROCESSED)
print("CACHE     :", CACHE)


repo_root : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager
OUTPUTS   : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\notebooks\strategy\overtake_probability\outputs
PROCESSED : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\processed\overtake_labeled
CACHE     : c:\Users\victo\Desktop\Documents\Cuarto Año\TFG\F1_Strat_Manager\data\cache\fastf1
